# Sort Processed Files

Moves calibrated/transform-corrected FITS files out of a Lights folder into per-type subfolders under a Processed folder, based on filename suffix:

| Suffix | Destination |
|---|---|
| `_cal` | `Processed/cal/` |
| `_TG` | `Processed/TG/` |
| `_TR` | `Processed/TR/` |
| `_TB` | `Processed/TB/` |

Only the top level of the Lights folder is scanned (no recursion). `.fit`/`.fits` files (any case) that don't match one of these suffixes are left in place. This is a **move**, not a copy -- `DRY_RUN` defaults to `True` so you can preview what would move before actually doing it; set it to `False` to run for real.

In [1]:
from pathlib import Path
import shutil

In [2]:
LIGHTS_PATH = "../../../Documents/astrophotography/Photometry/CzeV616/Light/"
PROCESSED_PATH = "../../../Documents/astrophotography/Photometry/CzeV616/Processed"

# Preview what would move without touching any files -- set False to actually move them.
DRY_RUN = False

In [3]:
SUFFIX_TO_SUBDIR = {
    "_cal": "cal",
    "_TG": "TG",
    "_TR": "TR",
    "_TB": "TB",
}
FITS_EXTENSIONS = {".fit", ".fits"}


def sort_processed_files(lights_path, processed_path, dry_run=True):
    lights_path = Path(lights_path)
    processed_path = Path(processed_path)

    moved = {subdir: [] for subdir in SUFFIX_TO_SUBDIR.values()}
    ignored = []
    skipped_existing = []

    for f in sorted(lights_path.iterdir()):
        if not f.is_file() or f.suffix.lower() not in FITS_EXTENSIONS:
            continue

        subdir = next((s for suffix, s in SUFFIX_TO_SUBDIR.items() if f.stem.endswith(suffix)), None)
        if subdir is None:
            ignored.append(f.name)
            continue

        dest_dir = processed_path / subdir
        dest = dest_dir / f.name
        if dest.exists():
            # Don't silently overwrite an existing file at the destination.
            skipped_existing.append(f.name)
            continue

        moved[subdir].append(f.name)
        if not dry_run:
            dest_dir.mkdir(parents=True, exist_ok=True)
            shutil.move(str(f), str(dest))

    return moved, ignored, skipped_existing

In [4]:
moved, ignored, skipped_existing = sort_processed_files(LIGHTS_PATH, PROCESSED_PATH, dry_run=DRY_RUN)

suffix = " (dry run)" if DRY_RUN else ""
for subdir, files in moved.items():
    print(f"{subdir}: {len(files)} file(s){suffix}")
    for name in files:
        print(f"  {name}")

if skipped_existing:
    print(f"\nSkipped (already exists at destination): {len(skipped_existing)} file(s)")
    for name in skipped_existing:
        print(f"  {name}")

print(f"\nIgnored (no matching suffix): {len(ignored)} file(s)")
for name in ignored:
    print(f"  {name}")

cal: 37 file(s)
  Light_CzeV616_60.0s_Bin1_20261013-190238_264deg_0001_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-190452_264deg_0002_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-190553_264deg_0003_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-190706_264deg_0004_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-190806_264deg_0005_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-190916_264deg_0006_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-191017_264deg_0007_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-191118_264deg_0008_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-191219_264deg_0009_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-191320_264deg_0010_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-191421_264deg_0011_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-191521_264deg_0012_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-191622_264deg_0013_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-191723_264deg_0014_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-191824_264deg_0015_cal.fit
  Light_CzeV616_60.0s_Bin1_20261013-191